# 03 - Modelado Analítico - Catastro DNC

En este notebook se construye el modelo analítico a partir de las tablas refinadas de la Dirección Nacional de Catastro.

A partir del análisis exploratorio realizado en el notebook anterior, se decide construir un modelo estrella con múltiples tablas de hechos, separando dimensiones descriptivas y tablas principales del dominio.

El modelo se guardará en la zona curated del datalake:

`/cur/obligatorio_catastro`

El objetivo de esta etapa es dejar los datos preparados para la posterior creación de tablas Hive y para responder preguntas analíticas.

## 1. Inicialización del entorno Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    when,
    lit,
    concat_ws,
    monotonically_increasing_id,
    current_date
)
from pyspark.sql.types import IntegerType, LongType, DoubleType

spark = SparkSession.builder \
    .appName("Obligatorio Catastro - Modelado") \
    .enableHiveSupport() \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-05T03:15:42,722 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Definición de rutas

Se definen las rutas de entrada y salida del proceso de modelado.

La entrada corresponde a la zona refinada `/ref`, generada en el notebook anterior.  
La salida corresponde a la zona curated `/cur`, donde se almacenará el modelo analítico.

In [2]:
REF_PATH = "/ref/obligatorio_catastro/dnc_2026_05"
CUR_PATH = "/cur/obligatorio_catastro"

print("REF_PATH:", REF_PATH)
print("CUR_PATH:", CUR_PATH)

REF_PATH: /ref/obligatorio_catastro/dnc_2026_05
CUR_PATH: /cur/obligatorio_catastro


## 3. Carga de tablas refinadas

Se cargan las tablas refinadas desde HDFS. Estas tablas ya tienen encabezados asignados y fueron exploradas en el notebook de EDA.

## 4. Modelo analítico seleccionado

Se selecciona un modelo estrella con múltiples tablas de hechos.

Las tablas principales del dominio catastral son:

- `padrones_urbanos`
- `padrones_rurales`
- `lineas_de_construccion`
- `historico_de_valores`

Estas tablas concentran las medidas principales del análisis: áreas, valores catastrales, características constructivas y evolución histórica de valores.

Las tablas auxiliares se utilizarán como dimensiones descriptivas para enriquecer el análisis.

In [3]:
tablas_ref = [
    "departamentos",
    "localidades",
    "destinos",
    "categorias_de_construccion",
    "estados_de_conservacion",
    "cubiertas",
    "cielorrasos",
    "tipos_de_obra",
    "padrones_urbanos",
    "padrones_rurales",
    "lineas_de_construccion",
    "historico_de_valores"
]

dfs = {}

for tabla in tablas_ref:
    dfs[tabla] = spark.read.parquet(f"{REF_PATH}/{tabla}")
    print(f"Tabla cargada: {tabla}")

Tabla cargada: departamentos
Tabla cargada: localidades
Tabla cargada: destinos
Tabla cargada: categorias_de_construccion
Tabla cargada: estados_de_conservacion
Tabla cargada: cubiertas
Tabla cargada: cielorrasos
Tabla cargada: tipos_de_obra
Tabla cargada: padrones_urbanos
Tabla cargada: padrones_rurales
Tabla cargada: lineas_de_construccion
Tabla cargada: historico_de_valores


## 5. Creación de dimensiones

En esta sección se construyen las dimensiones del modelo analítico a partir de las tablas auxiliares refinadas.

Las dimensiones permiten enriquecer las tablas de hechos con descripciones legibles para el análisis.

Se crean las siguientes dimensiones:

- `dim_departamento`
- `dim_localidad`
- `dim_destino`
- `dim_categoria_construccion`
- `dim_estado_conservacion`
- `dim_cubierta`
- `dim_cielorraso`
- `dim_tipo_obra`

En el caso de `dim_cielorraso`, se incorpora el código `0`, detectado en el EDA como valor presente en `lineas_de_construccion` pero ausente en la tabla auxiliar original.

In [4]:
dim_departamento = (
    dfs["departamentos"]
    .select(
        col("codigo_departamento"),
        col("departamento")
    )
    .dropDuplicates()
)

dim_localidad = (
    dfs["localidades"]
    .select(
        col("codigo_departamento"),
        col("codigo_localidad"),
        col("localidad")
    )
    .dropDuplicates()
)

dim_destino = (
    dfs["destinos"]
    .select(
        col("codigo_destino").cast(IntegerType()).alias("codigo_destino"),
        col("destino")
    )
    .dropDuplicates()
)

dim_categoria_construccion = (
    dfs["categorias_de_construccion"]
    .select(
        col("codigo_categoria").cast(DoubleType()).alias("codigo_categoria"),
        col("categoria_construccion")
    )
    .dropDuplicates()
)

dim_estado_conservacion = (
    dfs["estados_de_conservacion"]
    .select(
        col("codigo_estado").cast(DoubleType()).alias("codigo_estado"),
        col("estado_conservacion")
    )
    .dropDuplicates()
)

dim_cubierta = (
    dfs["cubiertas"]
    .select(
        col("codigo_cubierta").cast(IntegerType()).alias("codigo_cubierta"),
        col("cubierta")
    )
    .dropDuplicates()
)

dim_tipo_obra = (
    dfs["tipos_de_obra"]
    .select(
        col("codigo_tipo_obra").cast(IntegerType()).alias("codigo_tipo_obra"),
        col("tipo_obra")
    )
    .dropDuplicates()
)

In [5]:
dim_cielorraso_original = (
    dfs["cielorrasos"]
    .select(
        col("codigo_cielorraso").cast(IntegerType()).alias("codigo_cielorraso"),
        col("cielorraso")
    )
)

dim_cielorraso_cero = spark.createDataFrame(
    [(0, "Sin cielorraso / No informado")],
    ["codigo_cielorraso", "cielorraso"]
)

dim_cielorraso = (
    dim_cielorraso_original
    .unionByName(dim_cielorraso_cero)
    .dropDuplicates(["codigo_cielorraso"])
)

In [6]:
dimensiones = {
    "dim_departamento": dim_departamento,
    "dim_localidad": dim_localidad,
    "dim_destino": dim_destino,
    "dim_categoria_construccion": dim_categoria_construccion,
    "dim_estado_conservacion": dim_estado_conservacion,
    "dim_cubierta": dim_cubierta,
    "dim_cielorraso": dim_cielorraso,
    "dim_tipo_obra": dim_tipo_obra
}

for nombre, df in dimensiones.items():
    print("=" * 80)
    print(nombre)
    print("Registros:", df.count())
    df.show(10, truncate=False)

dim_departamento


Registros: 19
+-------------------+--------------+
|codigo_departamento|departamento  |
+-------------------+--------------+
|N                  |FLORES        |
|H                  |SALTO         |
|D                  |TREINTA Y TRES|
|I                  |PAYSANDU      |
|F                  |RIVERA        |
|P                  |LAVALLEJA     |
|L                  |COLONIA       |
|R                  |TACUAREMBO    |
|J                  |RIO NEGRO     |
|O                  |FLORIDA       |
+-------------------+--------------+
only showing top 10 rows

dim_localidad
Registros: 498
+-------------------+----------------+-------------+
|codigo_departamento|codigo_localidad|localidad    |
+-------------------+----------------+-------------+
|B                  |NG              |PIRIAPOLIS   |
|C                  |ME              |PEDREGAL     |
|M                  |JX              |A. NACIONAL  |
|A                  |BO              |EMPALME      |
|H                  |RA              |RINC

Registros: 2
+-----------------+-----------------------------+
|codigo_cielorraso|cielorraso                   |
+-----------------+-----------------------------+
|0                |Sin cielorraso / No informado|
|1                |Con cielorraso               |
+-----------------+-----------------------------+

dim_tipo_obra
Registros: 26
+----------------+-------------------------+
|codigo_tipo_obra|tipo_obra                |
+----------------+-------------------------+
|36              |Habilitada sin terminar  |
|50              |A Demoler                |
|34              |Habilitada sin terminar  |
|24              |Paralizada (mas de 1 a�o)|
|28              |Paralizada (m?s de 1 a�o)|
|23              |Paralizada (mas de 1 a�o)|
|15              |Reforma                  |
|37              |Habilitada sin terminar  |
|26              |Paralizada (mas de 1 a�o)|
|40              |A Construir              |
+----------------+-------------------------+
only showing top 10 rows



## 6. Creación de tablas de hechos

En esta sección se construyen las tablas de hechos del modelo estrella.

Las tablas de hechos contienen las medidas principales del dominio catastral y las claves necesarias para relacionarse con las dimensiones.

Se crean las siguientes tablas:

- `fact_padrones_urbanos`
- `fact_padrones_rurales`
- `fact_lineas_construccion`
- `fact_historico_valores`

Durante esta etapa se convierten columnas numéricas desde `string` a tipos adecuados (`int`, `long`, `double`) y se aplican reglas detectadas durante el EDA.

### 6.1 Tabla de hechos: padrones urbanos

La tabla `fact_padrones_urbanos` representa los padrones urbanos registrados por Catastro.

Esta tabla contiene medidas relevantes para el análisis, como:

- Área del predio.
- Área edificada.
- Valor catastral del terreno.
- Valor catastral de mejoras.
- Valor catastral total.
- Valor para impuestos.

Además, conserva las claves necesarias para relacionarse con las dimensiones de departamento y localidad.

In [7]:
fact_padrones_urbanos = (
    dfs["padrones_urbanos"]
    .select(
        col("codigo_regimen"),
        col("codigo_departamento"),
        col("codigo_localidad"),
        col("nro_padron").cast(IntegerType()).alias("nro_padron"),
        col("block_manzana"),
        col("ep_ss"),
        col("unidad").cast(IntegerType()).alias("unidad"),
        col("area_predio").cast(LongType()).alias("area_predio"),
        col("area_edificada").cast(LongType()).alias("area_edificada"),
        col("valor_catastral_terreno").cast(LongType()).alias("valor_catastral_terreno"),
        col("valor_catastral_mejoras").cast(LongType()).alias("valor_catastral_mejoras"),
        col("valor_catastral_total").cast(LongType()).alias("valor_catastral_total"),
        col("valor_para_impuestos").cast(LongType()).alias("valor_para_impuestos"),
        col("fecha_ultima_djcu"),
        col("vigencia_ultima_djcu")
    )
)

### 6.2 Tabla de hechos: padrones rurales

La tabla `fact_padrones_rurales` representa los padrones rurales registrados por Catastro.

A diferencia de los padrones urbanos, los padrones rurales se identifican mediante departamento, sección catastral y número de padrón.

Las principales medidas analíticas son:

- Área del predio.
- Valor catastral total.
- Valor para impuestos.

In [8]:
fact_padrones_rurales = (
    dfs["padrones_rurales"]
    .select(
        col("codigo_departamento"),
        col("nro_padron").cast(IntegerType()).alias("nro_padron"),
        col("seccion_catastral").cast(IntegerType()).alias("seccion_catastral"),
        col("area_predio").cast(LongType()).alias("area_predio"),
        col("valor_catastral_total").cast(LongType()).alias("valor_catastral_total"),
        col("valor_para_impuestos").cast(LongType()).alias("valor_para_impuestos")
    )
)

### 6.3 Tabla de hechos: líneas de construcción

La tabla `fact_lineas_construccion` representa las líneas constructivas asociadas a padrones urbanos.

Esta tabla permite analizar características de las construcciones, como:

- Destino de la construcción.
- Categoría constructiva.
- Estado de conservación.
- Tipo de cubierta.
- Indicador de cielorraso.
- Tipo de obra.
- Área construida.
- Año de construcción.

A partir del EDA se definieron reglas de limpieza para esta tabla:

- `anio_construccion = 0` se interpreta como año no informado.
- `anio_construccion > 2026` se interpreta como inconsistente.
- `anio_remanente = 0` se interpreta como no informado.
- `anio_remanente > 2026` se interpreta como inconsistente.
- `codigo_cielorraso = 0` se conserva para relacionarlo con la dimensión enriquecida `dim_cielorraso`.

In [9]:
ANIO_ACTUAL = 2026

fact_lineas_construccion = (
    dfs["lineas_de_construccion"]
    .select(
        col("codigo_regimen"),
        col("codigo_departamento"),
        col("codigo_localidad"),
        col("nro_padron").cast(IntegerType()).alias("nro_padron"),
        col("block_manzana"),
        col("ep_ss"),
        col("unidad").cast(IntegerType()).alias("unidad"),
        col("nivel").cast(DoubleType()).alias("nivel"),
        col("codigo_destino").cast(IntegerType()).alias("codigo_destino"),
        col("codigo_categoria").cast(DoubleType()).alias("codigo_categoria"),
        col("codigo_estado").cast(DoubleType()).alias("codigo_estado"),
        col("codigo_cubierta").cast(IntegerType()).alias("codigo_cubierta"),
        col("codigo_cielorraso").cast(IntegerType()).alias("codigo_cielorraso"),
        col("codigo_tipo_obra").cast(IntegerType()).alias("codigo_tipo_obra"),
        col("area_construida").cast(LongType()).alias("area_construida"),
        when(
            (col("anio_construccion").cast(IntegerType()) == 0) |
            (col("anio_construccion").cast(IntegerType()) > ANIO_ACTUAL),
            None
        ).otherwise(col("anio_construccion").cast(IntegerType())).alias("anio_construccion"),
        when(
            (col("anio_remanente").cast(IntegerType()) == 0) |
            (col("anio_remanente").cast(IntegerType()) > ANIO_ACTUAL),
            None
        ).otherwise(col("anio_remanente").cast(IntegerType())).alias("anio_remanente"),
        col("ep_ss_uso_exclusivo"),
        col("unidad_uso_exclusivo").cast(IntegerType()).alias("unidad_uso_exclusivo")
    )
)


### 6.4 Tabla de hechos: histórico de valores

La tabla `fact_historico_valores` representa la evolución reciente de los valores catastrales e impuestos asociados a los padrones.

Contiene valores correspondientes a años anteriores, permitiendo analizar variaciones históricas del valor catastral y del valor para impuestos.

Esta tabla conserva los identificadores necesarios para relacionarse con los padrones urbanos o rurales según el régimen correspondiente.

In [10]:
fact_historico_valores = (
    dfs["historico_de_valores"]
    .select(
        col("codigo_regimen"),
        col("codigo_departamento"),
        col("seccion_catastral").cast(IntegerType()).alias("seccion_catastral"),
        col("codigo_localidad"),
        col("nro_padron").cast(IntegerType()).alias("nro_padron"),
        col("block_manzana"),
        col("ep_ss"),
        col("unidad").cast(IntegerType()).alias("unidad"),
        col("valor_catastral_anio_1").cast(LongType()).alias("valor_catastral_anio_1"),
        col("valor_impuestos_anio_1").cast(LongType()).alias("valor_impuestos_anio_1"),
        col("valor_catastral_anio_2").cast(LongType()).alias("valor_catastral_anio_2"),
        col("valor_impuestos_anio_2").cast(LongType()).alias("valor_impuestos_anio_2"),
        col("valor_catastral_anio_3").cast(LongType()).alias("valor_catastral_anio_3"),
        col("valor_impuestos_anio_3").cast(LongType()).alias("valor_impuestos_anio_3"),
        col("valor_catastral_anio_4").cast(LongType()).alias("valor_catastral_anio_4"),
        col("valor_impuestos_anio_4").cast(LongType()).alias("valor_impuestos_anio_4")
    )
)

### 6.5 Validación de tablas de hechos

Se valida la creación de las tablas de hechos mostrando:

- Cantidad de registros.
- Esquema resultante.
- Primeras filas de cada tabla.

Esto permite comprobar que las conversiones de tipos y reglas de limpieza fueron aplicadas correctamente.

In [11]:
hechos = {
    "fact_padrones_urbanos": fact_padrones_urbanos,
    "fact_padrones_rurales": fact_padrones_rurales,
    "fact_lineas_construccion": fact_lineas_construccion,
    "fact_historico_valores": fact_historico_valores
}

for nombre, df in hechos.items():
    print("=" * 80)
    print(nombre)
    print("Registros:", df.count())
    df.printSchema()
    df.show(5, truncate=False)

fact_padrones_urbanos
Registros: 1468837
root
 |-- codigo_regimen: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- codigo_localidad: string (nullable = true)
 |-- nro_padron: integer (nullable = true)
 |-- block_manzana: string (nullable = true)
 |-- ep_ss: string (nullable = true)
 |-- unidad: integer (nullable = true)
 |-- area_predio: long (nullable = true)
 |-- area_edificada: long (nullable = true)
 |-- valor_catastral_terreno: long (nullable = true)
 |-- valor_catastral_mejoras: long (nullable = true)
 |-- valor_catastral_total: long (nullable = true)
 |-- valor_para_impuestos: long (nullable = true)
 |-- fecha_ultima_djcu: date (nullable = true)
 |-- vigencia_ultima_djcu: date (nullable = true)

+--------------+-------------------+----------------+----------+-------------+-----+------+-----------+--------------+-----------------------+-----------------------+---------------------+--------------------+-----------------+--------------------+
|codig

## 7. Guardado del modelo en zona curated

En esta sección se guardan las dimensiones y tablas de hechos en la zona curated del datalake.

La zona curated contiene el modelo analítico ya preparado para ser consumido posteriormente por Hive y por las consultas analíticas.

Cada dimensión y tabla de hechos se guarda en formato Parquet dentro de:

`/cur/obligatorio_catastro`

### 7.1 — Definir tablas del modelo

In [12]:
modelo_curated = {}

modelo_curated.update(dimensiones)
modelo_curated.update(hechos)

print("Tablas a guardar en curated:")
for nombre in modelo_curated.keys():
    print("-", nombre)

Tablas a guardar en curated:
- dim_departamento
- dim_localidad
- dim_destino
- dim_categoria_construccion
- dim_estado_conservacion
- dim_cubierta
- dim_cielorraso
- dim_tipo_obra
- fact_padrones_urbanos
- fact_padrones_rurales
- fact_lineas_construccion
- fact_historico_valores


### 7.2 — Guardar dimensiones y hechos en /cur

In [13]:
def guardar_parquet(df, nombre_tabla):
    output_path = f"{CUR_PATH}/{nombre_tabla}"
    print(f"Guardando {nombre_tabla} en {output_path}")
    
    (
        df.write
        .mode("overwrite")
        .parquet(output_path)
    )
    
    print(f"OK: {nombre_tabla}")

In [14]:
for nombre, df in dimensiones.items():
    guardar_parquet(df, nombre)

print("Dimensiones guardadas correctamente.")

Guardando dim_departamento en /cur/obligatorio_catastro/dim_departamento
OK: dim_departamento
Guardando dim_localidad en /cur/obligatorio_catastro/dim_localidad
OK: dim_localidad
Guardando dim_destino en /cur/obligatorio_catastro/dim_destino
OK: dim_destino
Guardando dim_categoria_construccion en /cur/obligatorio_catastro/dim_categoria_construccion
OK: dim_categoria_construccion
Guardando dim_estado_conservacion en /cur/obligatorio_catastro/dim_estado_conservacion
OK: dim_estado_conservacion
Guardando dim_cubierta en /cur/obligatorio_catastro/dim_cubierta
OK: dim_cubierta
Guardando dim_cielorraso en /cur/obligatorio_catastro/dim_cielorraso
OK: dim_cielorraso
Guardando dim_tipo_obra en /cur/obligatorio_catastro/dim_tipo_obra
OK: dim_tipo_obra
Dimensiones guardadas correctamente.


In [15]:
guardar_parquet(fact_padrones_urbanos, "fact_padrones_urbanos")

Guardando fact_padrones_urbanos en /cur/obligatorio_catastro/fact_padrones_urbanos


[Stage 124:>                                                        (0 + 2) / 2]

OK: fact_padrones_urbanos


In [16]:
guardar_parquet(fact_padrones_rurales, "fact_padrones_rurales")

Guardando fact_padrones_rurales en /cur/obligatorio_catastro/fact_padrones_rurales


[Stage 125:============================>                            (1 + 1) / 2]

OK: fact_padrones_rurales


In [17]:
guardar_parquet(fact_lineas_construccion, "fact_lineas_construccion")

Guardando fact_lineas_construccion en /cur/obligatorio_catastro/fact_lineas_construccion


[Stage 126:============================>                            (1 + 1) / 2]

OK: fact_lineas_construccion


In [18]:
guardar_parquet(fact_historico_valores, "fact_historico_valores")

Guardando fact_historico_valores en /cur/obligatorio_catastro/fact_historico_valores


[Stage 127:============================>                            (1 + 1) / 2]

OK: fact_historico_valores


## 8. Validación del modelo guardado

Se leen nuevamente las tablas desde la zona curated para verificar que fueron guardadas correctamente.

La validación incluye:

- Nombre de la tabla.
- Ruta en HDFS.
- Cantidad de registros.
- Cantidad de columnas.

Esta validación permite confirmar que el modelo analítico quedó persistido correctamente en HDFS.

In [19]:
from pyspark.sql import Row

validacion_curated = []

for nombre in modelo_curated.keys():
    path = f"{CUR_PATH}/{nombre}"
    df = spark.read.parquet(path)
    
    validacion_curated.append(
        Row(
            tabla=nombre,
            path=path,
            cantidad_registros=df.count(),
            cantidad_columnas=len(df.columns)
        )
    )

df_validacion_curated = spark.createDataFrame(validacion_curated)

df_validacion_curated.orderBy("tabla").show(truncate=False)

+--------------------------+----------------------------------------------------+------------------+-----------------+
|tabla                     |path                                                |cantidad_registros|cantidad_columnas|
+--------------------------+----------------------------------------------------+------------------+-----------------+
|dim_categoria_construccion|/cur/obligatorio_catastro/dim_categoria_construccion|9                 |2                |
|dim_cielorraso            |/cur/obligatorio_catastro/dim_cielorraso            |2                 |2                |
|dim_cubierta              |/cur/obligatorio_catastro/dim_cubierta              |5                 |2                |
|dim_departamento          |/cur/obligatorio_catastro/dim_departamento          |19                |2                |
|dim_destino               |/cur/obligatorio_catastro/dim_destino               |72                |2                |
|dim_estado_conservacion   |/cur/obligatorio_cat

## 9. Conclusiones del modelado

Se construyó un modelo estrella con múltiples tablas de hechos y dimensiones descriptivas.

El modelo quedó persistido en la zona curated del datalake, bajo la ruta:

`/cur/obligatorio_catastro`

Las tablas de hechos principales representan los padrones urbanos, padrones rurales, líneas de construcción e histórico de valores.  
Las dimensiones permiten enriquecer el análisis con información territorial y descriptiva, como departamentos, localidades, destinos de construcción, categorías constructivas, estados de conservación, cubiertas, cielorrasos y tipos de obra.

Durante el modelado se aplicaron reglas derivadas del EDA, entre ellas:

- Conversión de columnas numéricas a tipos adecuados.
- Tratamiento de años de construcción no informados o inconsistentes.
- Enriquecimiento de la dimensión `dim_cielorraso` incorporando el código `0` como "Sin cielorraso / No informado".
- Separación entre zona refinada y zona curated para mantener trazabilidad.

El modelo queda preparado para la creación de tablas Hive y para responder preguntas analíticas en la siguiente etapa.